# Homework 10
#### Course Notes
**Language Models:** https://github.com/rjenki/BIOS512/tree/main/lecture17  
**Unix:** https://github.com/rjenki/BIOS512/tree/main/lecture18  
**Docker:** https://github.com/rjenki/BIOS512/tree/main/lecture19

## Question 1
#### Make a language model that uses ngrams and allows the user to specify start words, but uses a random start if one is not specified.

#### a) Make a function to tokenize the text.

#### b) Make a function generate keys for ngrams.

#### c) Make a function to build an ngram table.

#### d) Function to digest the text.

#### e) Function to digest the url.

#### f) Function that gives random start.

#### g) Function to predict the next word.

#### h) Function that puts everything together. Specify that if the user does not give a start word, then the random start will be used.

## Question 2
#### For this question, set `seed=2025`.
#### a) Test your model using a text file of [Grimm's Fairy Tails](https://www.gutenberg.org/cache/epub/2591/pg2591.txt)
#### i) Using n=3, with the start word(s) "the king", with length=15. 
#### ii) Using n=3, with no start word, with length=15.

#### b) Test your model using a text file of [Ancient Armour and Weapons in Europe](https://www.gutenberg.org/cache/epub/46342/pg46342.txt)
#### i) Using n=3, with the start word(s) "the king", with length=15. 
#### ii) Using n=3, with no start word, with length=15.

#### c) Explain in 1-2 sentences the difference in content generated from each source.

## Question 3
#### a) What is a language learning model? 
#### b) Imagine the internet goes down and you can't run to your favorite language model for help. How do you run one locally?

## Question 4
#### Explain what the following vocab words mean in the context of typing `mkdir project` into the command line. If the term doesn't apply to this command, give the definition and/or an example.
| Term | Meaning |  
|------|---------|
| **Shell** |  |
| **Terminal emulator** |  |
| **Process** |  |
| **Signal** |  |
| **Standard input** |  |
| **Standard output** |  |
| **Command line argument** |  |
| **The environment** |  |

## Question 5
#### Consider the following command `find . -iname "*.R" | xargs grep read_csv`.
#### a) What are the programs?
#### b) Explain what this command is doing, part by part.

## Question 6
#### Install Docker on your machine. See [here](https://github.com/rjenki/BIOS512/blob/main/lecture18/docker_install.md) for instructions. 
#### a) Show the response when you run `docker run hello-world`.
#### b) Access Rstudio through a Docker container. Set your password and make sure your files show up on the Rstudio server. Type the command and the output you get below.
#### c) How do you log in to the RStudio server?

### Question 1 (parts e–h) — R solutions appended here
This section contains R functions that implement the requested functionality for Question 1:
- **e)** `digest_url(url)` — fetch & extract text (handles raw `.ipynb` JSON and plain HTML/text), then tokenize.
- **f)** `random_start(model, n)` — sample a random context suitable for the model order.
- **g)** `predict_next_word(model, context, deterministic=FALSE)` — sample or deterministically select next word.
- **h)** `build_and_generate(text_or_url, n, length, start=NULL, deterministic=FALSE)` — end-to-end wrapper.
Paste and run the R code cells in an R kernel or RStudio notebook cell.


In [ ]:

# --- Shared helpers (tokenize + ngram builder) ---
library(stringr)
library(httr)
library(jsonlite)

tokenize <- function(text) {
  txt <- tolower(text)
  txt <- str_replace_all(txt, "\n", " ")
  toks <- str_extract_all(txt, "[a-z0-9']+|[^\\s\\w]")[[1]]
  toks
}

gen_keys <- function(tokens, n) {
  pairs <- list()
  if (n < 1) stop("n must be >= 1")
  if (n == 1) {
    for (i in seq_along(tokens)) pairs[[length(pairs) + 1]] <- list(ctx = character(0), next = tokens[i])
    return(pairs)
  }
  if (length(tokens) < n) return(pairs)
  for (i in seq_len(length(tokens) - n + 1)) {
    ctx <- tokens[i:(i + n - 2)]
    nxt <- tokens[i + n - 1]
    pairs[[length(pairs) + 1]] <- list(ctx = ctx, next = nxt)
  }
  pairs
}

build_ngram_model <- function(tokens, n) {
  model <- list()
  pairs <- gen_keys(tokens, n)
  for (p in pairs) {
    key <- if (length(p$ctx) == 0) "" else paste(p$ctx, collapse = " ")
    if (is.null(model[[key]])) model[[key]] <- integer(0)
    existing <- model[[key]]
    if (is.na(existing[p$next])) existing[p$next] <- 0L
    existing[p$next] <- existing[p$next] + 1L
    model[[key]] <- existing
  }
  model
}


In [ ]:

# --- e) digest_url(url) ---
digest_url <- function(url, user_agent = "R-lang") {
  resp <- httr::GET(url, httr::user_agent(user_agent))
  if (httr::http_error(resp)) stop("Failed to GET: ", url, " (", httr::status_code(resp), ")")
  body_text <- httr::content(resp, as = "text", encoding = "UTF-8")
  content_type <- httr::headers(resp)[["content-type"]]
  # Try parse as ipynb JSON if possible
  if (!is.null(content_type) && grepl("application/json", content_type, ignore.case = TRUE)) {
    j <- tryCatch(jsonlite::fromJSON(body_text), error = function(e) NULL)
    if (!is.null(j) && !is.null(j$cells)) {
      pieces <- vapply(j$cells, function(cell) {
        if (!is.null(cell$source)) paste(unlist(cell$source), collapse = " ") else ""
      }, FUN.VALUE = "")
      longtext <- paste(pieces, collapse = " ")
      return(tokenize(longtext))
    }
  }
  # fallback: if body_text contains "cells" key, attempt parse anyway
  if (grepl('"cells"', body_text)) {
    j <- tryCatch(jsonlite::fromJSON(body_text), error = function(e) NULL)
    if (!is.null(j) && !is.null(j$cells)) {
      pieces <- vapply(j$cells, function(cell) {
        if (!is.null(cell$source)) paste(unlist(cell$source), collapse = " ") else ""
      }, FUN.VALUE = "")
      longtext <- paste(pieces, collapse = " ")
      return(tokenize(longtext))
    }
  }
  # Remove script/style and HTML tags crudely
  plain <- gsub("<script.*?>.*?</script>", " ", body_text, perl = TRUE)
  plain <- gsub("<style.*?>.*?</style>", " ", plain, perl = TRUE)
  plain <- gsub("<[^>]+>", " ", plain)
  plain <- gsub("\s+", " ", plain)
  tokenize(plain)
}


In [ ]:

# --- f) random_start(model, n) ---
random_start <- function(model, n) {
  if (n == 1) return(character(0))
  keys <- names(model)
  if (length(keys) == 0) stop("Model has no contexts")
  attempts <- 0
  while (attempts < 200) {
    k <- sample(keys, 1)
    toks <- if (nzchar(k)) str_split(k, " ", simplify = TRUE) else character(0)
    if (length(toks) == n - 1) return(as.character(toks))
    attempts <- attempts + 1
  }
  for (k in keys) {
    toks <- if (nzchar(k)) str_split(k, " ", simplify = TRUE) else character(0)
    if (length(toks) >= n - 1) return(as.character(tail(toks, n - 1)))
  }
  rep("<s>", n - 1)
}


In [ ]:

# --- g) predict_next_word(model, context, deterministic = FALSE) ---
predict_next_word <- function(model, context, deterministic = FALSE) {
  ctx_key <- if (length(context) == 0) "" else paste(as.character(context), collapse = " ")
  counts <- model[[ctx_key]]
  if (is.null(counts) || length(counts) == 0) {
    fallback_key <- sample(names(model), 1)
    counts <- model[[fallback_key]]
  }
  words <- names(counts)
  freqs <- as.numeric(counts)
  if (deterministic) {
    words[which.max(freqs)]
  } else {
    probs <- freqs / sum(freqs)
    sample(words, size = 1, prob = probs)
  }
}


In [ ]:

# --- h) build_and_generate(text_or_url, n = 3, length = 50, start = NULL, deterministic = FALSE) ---
build_and_generate <- function(text_or_url, n = 3, length = 50, start = NULL, deterministic = FALSE) {
  tokens <- if (is.character(text_or_url) && grepl("^https?://", text_or_url)) {
    digest_url(text_or_url)
  } else if (is.character(text_or_url)) {
    tokenize(text_or_url)
  } else stop("text_or_url must be a URL or a character string of text")
  
  model <- build_ngram_model(tokens, n)
  if (n == 1) context <- character(0) else {
    if (!is.null(start)) {
      st_toks <- tokenize(start)
      if (length(st_toks) >= n - 1) {
        context <- as.character(tail(st_toks, n - 1))
      } else {
        context <- c(rep("<s>", (n - 1) - length(st_toks)), st_toks)
      }
    } else {
      context <- random_start(model, n)
    }
  }
  out <- if (n == 1) character(0) else as.character(context)
  for (i in seq_len(length)) {
    nxt <- predict_next_word(model, context, deterministic = deterministic)
    out <- c(out, nxt)
    if (n == 1) {
      context <- character(0)
    } else {
      context <- as.character(tail(c(context, nxt), n - 1))
    }
  }
  paste(out, collapse = " ")
}
# --- end of functions ---
